# *E. coli* Fatty Acid Synthesis — Deterministic ODE Model

Simulates the type II FAS pathway using mass-action kinetics. Reaction networks are defined in YAML files and loaded at runtime; the resulting ODE system is solved with JAX + diffrax.

**Workflow:** build network → reaction sanity check → set initial conditions → solve ODEs → make plots

---
*Annette Thompson · Fox and Shirts Labs, CU Boulder · 2026 · Developed with assistance from GitHub Copilot (Claude Sonnet 4.6)*

## Step 1: Load Reaction Network

**What to update before running:**
- **`REACTIONS_PATH`** — set this to the folder containing your YAML reaction files. Update to `Path('../Reactions/EC_FAS_ME1')` once the full YAML folder is in place.

`build_ode_system_from_reactions()` returns:
- `network` — equinox module implementing the ODE right-hand side
- `species` — ordered list of species names
- `params` — ordered list of parameter names
- `param_values` — dict of default rate constants loaded from the YAML files
- `scaling_groups` — named groups of rate constants that can be rescaled together

The `sp`, `pm`, `rn`, and `sg` namespace objects enable **tab-completion**: type `sp.` and press Tab to browse all species, `pm.` for parameters, etc.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../")

from pathlib import Path
import re
import os

import numpy as np
import matplotlib.pyplot as plt
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import diffrax as dfrx
import pandas as pd
import itertools

from Utilities.reaction_model_builder import (
    build_ode_system_from_reactions,
    load_elementary_reactions,
    make_namespace,
    make_reaction_namespace,
    set_scaling_group_values
)

REACTIONS_PATH = Path('../Reactions/EC_FAS_ME1')

network, species, params, param_values, scaling_groups = build_ode_system_from_reactions(REACTIONS_PATH)
rxns = load_elementary_reactions(REACTIONS_PATH)

# Namespaces enable tab-completion: sp.ACP, pm.k3_1f, rn.FabG_binding_NADPH
sp = make_namespace(species)
pm = make_namespace(params)
rn = make_reaction_namespace(rxns)
sg = make_namespace(scaling_groups)

theta = jnp.array([param_values[p] for p in params], dtype=jnp.float64)

print(f'Loaded {len(species)} species, {len(params)} params, {len(rxns)} reactions, and {len(scaling_groups)} scaling groups.')

## Step 2: Sanity Checks

Checks reaction definitions for common errors before committing to a long simulation:
- Mass balance (atoms conserved across each reaction)
- Species continuity (every produced species is also consumed somewhere)
- Chain-set consistency (for fatty acid chain-length templates)
- Suspiciously similar rate-constant names (typo detection)
- Zero-valued rate constants

In [ ]:
import importlib, Utilities.reaction_sanity_check
importlib.reload(Utilities.reaction_sanity_check)
from Utilities.reaction_sanity_check import run_all_checks

sanity = run_all_checks(rxns)

## Tools for Inspecting Reactions

Use these helpers to debug reaction definitions and quickly inspect model structure.

`rxns.query(query, query_type, *, return_type="expression")` searches reactions by field.

- `query_type` (required): `"species"`, `"reactant"`, `"product"`, `"parameter"`, `"scaling"`, or any reaction attribute (for example `"rxn_name"`)
- `query`: substring text (contains match) or a namespace index like `sp.X`, `pm.k`, `sg.a1` (exact match); can use Regular Expression to search for patterns (requires use_regex = True)
- `return_type` (optional): `"expression"` (default), `"parameters"` (includes parameter values), `"scaling"`, `"ode_terms"`, or `"scaled_rate_constants"` (effective rate = base × scaling factor; requires `theta=theta`)

`rxns.get_ode_terms(sp.X)` is a shortcut for the species mass-balance view.

In [ ]:
rxns.query("a1", "scaling", return_type="parameters",use_regex=True)

## Step 3: Simulation Parameters

**What to change here:**
- **`y0[sp.X] = value`** — starting concentration (µM) of each species (substrates).
- **`enzyme_concs`** — enzyme starting concentrations (µM).
- **`SCALING_GROUPS`** — scalar multipliers applied uniformly to all rate constants in a named group (groups set in reaction files).

In [ ]:
y0 = np.zeros(len(species), dtype=np.float64)

# Substrates (µM) - set non-zero starting concentrations 
# (NADPH and NADH are typically set to 1000 uM, 
# other concentrations should align with in vitro experiments)
y0[sp.C2_AcCoA]  = 500.0
y0[sp.C3_MalCoA] = 500.0
y0[sp.ACP]       = 10.0
y0[sp.NADPH]     = 1000.0
y0[sp.NADH]      = 1000.0

# Enzymes (µM)
enzyme_concs = {
    sp.FabD: 1, sp.FabH: 1, sp.FabG: 1, sp.FabZ: 1,
    sp.FabI: 1, sp.TesA: 10, sp.FabF: 1, sp.FabA: 1, sp.FabB: 1,
}
for idx, conc in enzyme_concs.items():
    y0[idx] = float(conc)

# Scaling group values - setting to 1 or 0 will leave parameters alone 
# depending on the scaling parameter (some are within expressions used to scale)
SCALING_GROUPS = {
    "a1": 1, "a2": 1, "a3": 1,
    "b1": 1, "b2": 1, "b3": 1,
    "c1": 1, "c2": 1, "c3": 1, "c4": 1,
    "d1": 0, "d2": 0, "e": 1, "f": 1,
    "x1": 1, "x2": 1, "x3": 1, "x4": 1,
}
theta = set_scaling_group_values(theta, params, SCALING_GROUPS)

## Step 4: Solve ODEs

**What to change here:**
- **`time_range = [0.0, 720.0]`** — simulation start and end time in **seconds** (720 s = 12 min (product profile), 150 s = 2.5 min (initial rate)). Increase the upper bound to let the system approach steady state.

The solver used is **Kvaerno5** — an implicit, stiff-safe 5th-order Runge–Kutta method with adaptive step size via a PID controller. Every accepted step is saved; trailing NaN padding is trimmed after solving. Time units are **seconds** internally.

In [ ]:
time_range = [0.0, 150.0]  # seconds

def run_odes(time_range, y0, max_accepted_steps=None, max_total_steps=None):
    capped_mode = (max_accepted_steps is not None) or (max_total_steps is not None)

    # Diffrax max_steps limits total attempted steps (accepted + rejected).
    if max_total_steps is not None:
        internal_max_steps = int(max_total_steps)
    elif max_accepted_steps is not None:
        internal_max_steps = int(max(max_accepted_steps * 5, max_accepted_steps))
    else:
        internal_max_steps = 10_000

    sol = dfrx.diffeqsolve(
        dfrx.ODETerm(network),
        dfrx.Kvaerno5(),
        t0=time_range[0], t1=time_range[1], dt0=1e-6,
        y0=jnp.asarray(y0, dtype=jnp.float64),
        args=theta,
        saveat=dfrx.SaveAt(steps=True),
        stepsize_controller=dfrx.PIDController(
            rtol=1e-5, atol=1e-8, pcoeff=0.2, icoeff=0.4, dcoeff=0,
        ),
        max_steps=internal_max_steps,
        throw=False,
    )

    raw_num_ac_steps = int(np.asarray(sol.stats['num_accepted_steps']))
    num_steps = int(np.asarray(sol.stats['num_steps']))

    T_full = sol.ts[:raw_num_ac_steps]
    C_full = sol.ys[:raw_num_ac_steps, :]

    if raw_num_ac_steps == 0:
        t_final_full = float(time_range[0])
    else:
        t_final_full = float(T_full[-1])

    if capped_mode:
        completed_full = bool(t_final_full >= float(time_range[1]) - 1e-9)

        hit_accepted_cap = bool(
            (max_accepted_steps is not None)
            and (raw_num_ac_steps >= int(max_accepted_steps))
            and (not completed_full)
        )

        hit_total_cap = bool(
            (max_total_steps is not None)
            and (num_steps >= int(max_total_steps))
            and (not completed_full)
        )

        # If accepted-step cap is reached first, trim returned trajectory to that limit.
        if hit_accepted_cap:
            num_ac_steps = int(max_accepted_steps)
            T = T_full[:num_ac_steps]
            C = C_full[:num_ac_steps, :]
            t_final = float(T[-1]) if len(T) else float(time_range[0])
            completed = False
        else:
            num_ac_steps = raw_num_ac_steps
            T = T_full
            C = C_full
            t_final = t_final_full
            completed = completed_full

        if (hit_accepted_cap or hit_total_cap) and (not completed):
            hit_labels = []
            if hit_accepted_cap:
                hit_labels.append(f"accepted-step cap ({int(max_accepted_steps)})")
            if hit_total_cap:
                hit_labels.append(f"total-step cap ({int(max_total_steps)})")
            print(
                f"Stopped at {' and '.join(hit_labels)} before t1={float(time_range[1]):.1f} s. "
                f"Reached t_final={t_final:.1f} s"
            )
        else:
            print(f"Solved in {num_ac_steps} accepted steps, {num_steps} total steps.  t_final = {t_final:.1f} s")

        stats = {
            "num_steps": num_steps,
            "num_accepted_steps": num_ac_steps,
            "raw_num_accepted_steps": raw_num_ac_steps,
            "t_final": t_final,
            "completed": completed,
            "hit_accepted_cap": hit_accepted_cap,
            "hit_total_cap": hit_total_cap,
        }
        return T, C, stats

    else:
        num_ac_steps = raw_num_ac_steps
        T = T_full
        C = C_full
        t_final = t_final_full

        print(f"Solved in {num_ac_steps} accepted steps, {num_steps} total steps.  t_final = {t_final:.1f} s")
        return T, C


T, C = run_odes(time_range, y0)

## Results

### Fatty Acid Chain-Length Profile

Stacked bar chart of final (end-of-simulation) fatty acid mole fractions, split by saturated vs. unsaturated. Species matching `C<n>_FA` and `C<n>_FA_un` are collected and totaled across all chain lengths.

In [ ]:
# Collect final concentrations of all C<n>_FA (sat) and C<n>_FA_un (unsat) species
SP_PATTERN = re.compile(r'^C(\d+)_FA(_unsat)?$')
sat_by_chain: dict[int, float] = {}
unsat_by_chain: dict[int, float] = {}
final_conc = C[-1, :]

for sp_name in species:
    m = SP_PATTERN.match(sp_name)
    if m is None:
        continue
    chain = int(m.group(1))
    val = float(final_conc[getattr(sp, sp_name)])
    (unsat_by_chain if m.group(2) else sat_by_chain)[chain] = \
        (unsat_by_chain if m.group(2) else sat_by_chain).get(chain, 0.0) + val

chains = sorted(set(sat_by_chain) | set(unsat_by_chain))
sat   = np.array([sat_by_chain.get(c, 0.0)   for c in chains])
unsat = np.array([unsat_by_chain.get(c, 0.0) for c in chains])
tot = sat.sum() + unsat.sum()

# Normalize to mole fractions (avoid divide-by-zero if all zero)
sat_frac   = np.divide(sat,   tot, out=np.zeros_like(sat),   where=tot > 0)
unsat_frac = np.divide(unsat, tot, out=np.zeros_like(unsat), where=tot > 0)

plt.style.use('seaborn-v0_8-poster')
plt.rcParams['axes.prop_cycle'] = plt.cycler(
    color=[plt.cm.gist_earth(i / 4) for i in range(4)]
)

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(chains))
ax.bar(x, sat_frac,   label='Saturated')
ax.bar(x, unsat_frac, bottom=sat_frac, label='Unsaturated')
ax.set_xticks(x)
ax.set_xticklabels([str(c) for c in chains])
ax.set_xlabel('FA Chain Length')
ax.set_ylabel('Mole Fraction')
ax.set_title('Fatty Acid Profile at t = {:.0f} s'.format(float(T[-1])))
ax.legend()
plt.show()


### Concentration Trajectories

`plot_species(names)` accepts any mix of `sp.<name>` integer indices or plain string names. Pass `log_y=True` for species that span several orders of magnitude.

In [ ]:
def plot_species(names: list, *, log_y: bool = False) -> None:
    """Plot concentration vs time. Accepts sp.<name> indices or string names."""
    fig, ax = plt.subplots(figsize=(9, 5))
    for name in names:
        idx, label = (name, species[name]) if isinstance(name, int) else (getattr(sp, name), name)
        ax.plot(T / 60, C[:, idx], label=label)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Concentration (µM)')
    if log_y:
        ax.set_yscale('log')
    ax.legend()
    fig.tight_layout()
    plt.show()

# CoA-linked substrates
plot_species([sp.C3_MalACP])


### Export to experiment csvs

Quick write up to make data files for running Bayesian inference

In [ ]:
# Functions to make timeseries and endpoint CSVs

def export_species_time_course(time_range, y0, species_idx, folder_name, save_csv=True):

    T, C = run_odes(time_range, y0)
    
    species_data = C[:, species_idx]
    idx_to_name = {v: k for k, v in vars(sp).items()}
    species_name = idx_to_name.get(species_idx, f"Species_{species_idx}")
    
    columns = ["Time (s)", f"{species_name} (uM)"]
    df = pd.DataFrame(np.vstack([[0,0], np.column_stack([T, species_data])]), columns=columns)
    
    # Shorten number of rows
    keep_rows = [0]
    last_accepted_time = df.loc[0, "Time (s)"]
    
    for i in range(1, len(df)):
        current_time = df.loc[i, "Time (s)"]
        if current_time - last_accepted_time >= 5:
            keep_rows.append(i)
            last_accepted_time = current_time
            
    df_filtered = df.iloc[keep_rows].reset_index(drop=True)
    
    filename = f"../Data/{folder_name}/time_vs_{species_name}.csv"
    if save_csv:
        df_filtered.to_csv(filename, index=False)
        print(f"Saved to {filename}")
    
    print("Run complete!")
    return df_filtered


def sweep_concentrations(time_range, base_y0, varying_species, target_species_idx, folder_name, save_csv=True, max_accepted_steps=None, max_total_steps=None):

    idx_to_name = {v: k for k, v in vars(sp).items()}
    target_name = idx_to_name.get(target_species_idx, f"Species_{target_species_idx}")
    
    species_indices = list(varying_species.keys())
    concentration_lists = list(varying_species.values())
    
    columns = [f"{idx_to_name.get(idx, f'Species_{idx}')} (uM)" for idx in species_indices]

    results = []

    capped_mode = (max_accepted_steps is not None) or (max_total_steps is not None)

    if capped_mode:
        columns += [
            f"{target_name} (uM)",
            "Accepted Steps",
            "Total Steps",
            "t_final (s)",
            "Completed",
            "Hit Accepted-Step Cap",
            "Hit Total-Step Cap",
        ]
        
        for combo in itertools.product(*concentration_lists):
            y0_run = base_y0.copy()
            
            for idx, current_conc in zip(species_indices, combo):
                y0_run[idx] = float(current_conc)
                
            T, C, stats = run_odes(
                time_range,
                y0_run,
                max_accepted_steps=max_accepted_steps,
                max_total_steps=max_total_steps,
            )

            if len(C) == 0:
                final_val = np.nan
            else:
                final_val = float(C[-1, target_species_idx])
            
            results.append(
                list(combo)
                + [
                    final_val,
                    stats["num_accepted_steps"],
                    stats["num_steps"],
                    stats["t_final"],
                    stats["completed"],
                    stats["hit_accepted_cap"],
                    stats["hit_total_cap"],
                ]
            )
        
    else:
        columns += [f"{target_name} (uM)"]
        
        for combo in itertools.product(*concentration_lists):
            y0_run = base_y0.copy()
            
            for idx, current_conc in zip(species_indices, combo):
                y0_run[idx] = float(current_conc)
                
            T, C = run_odes(time_range, y0_run)

            final_val = float(C[-1, target_species_idx])
            results.append(list(combo) + [final_val])
        
    df_sweep = pd.DataFrame(results, columns=columns)

    filename = f"../Data/{folder_name}/conc_vs_final_{target_name}.csv"
    if save_csv:
        df_sweep.to_csv(filename, index=False)
        print(f"Saved to {filename}")
        
    print("Sweep complete!")
    return df_sweep

In [ ]:
# Experiment 1 - MalACP v time

enzyme_name = "FabD"
os.makedirs(f"../Data/{enzyme_name}", exist_ok=True)

# Base y0
y0 = np.zeros(len(species), dtype=np.float64)
y0[sp.C3_MalCoA] = 0.1
y0[sp.ACP]       = 10
enzyme_concs = {
    sp.FabD: 0.001
}
for idx, conc in enzyme_concs.items():
    y0[idx] = float(conc)

time_range = [0,150]
df_filtered = export_species_time_course(time_range, y0, sp.C3_MalACP, enzyme_name, save_csv=False)

In [ ]:
# Experiment 2 - initial_conc vs final MalACP

time_range = [0, 720]
max_accepted_steps = None
max_total_steps = None

my_sweep_config = {
    sp.FabD: [1e-3, 1e-2],
    sp.C3_MalCoA: [0.1, 1],
    sp.ACP: [10, 100],
}

df_res = sweep_concentrations(
    time_range=time_range,
    base_y0=y0,
    varying_species=my_sweep_config,
    target_species_idx=sp.C3_MalACP,
    folder_name=enzyme_name,
    save_csv=False,
    max_accepted_steps=max_accepted_steps,
    max_total_steps=max_total_steps,
)

if (max_accepted_steps is not None) or (max_total_steps is not None):
    fastest_8 = (
        df_res
        .sort_values(["Accepted Steps", "Total Steps", "t_final (s)"], ascending=[True, True, False])
        .head(8)
        .reset_index(drop=True)
    )

    fastest_8_completed = (
        df_res[df_res["Completed"]]
        .sort_values(["Accepted Steps", "Total Steps", "t_final (s)"], ascending=[True, True, False])
        .head(8)
        .reset_index(drop=True)
    )

    print(f"Completed runs: {int(df_res['Completed'].sum())}/{len(df_res)}")
    print("\nTop 8 runs (least accepted/total steps, highest t_final):")
    display(fastest_8)

    print("\nTop 8 completed runs (least accepted/total steps, highest t_final):")
    display(fastest_8_completed)

    idx_to_name = {v: k for k, v in vars(sp).items()}
    species_indices = list(my_sweep_config.keys())
    species_cols = [f"{idx_to_name.get(idx, f'Species_{idx}')} (uM)" for idx in species_indices]

    def pick_best_two_concentrations_per_species(df, species_cols):
        value_pairs_by_col = {}
        for col in species_cols:
            values = sorted(df[col].dropna().unique().tolist())
            value_pairs_by_col[col] = list(itertools.combinations(values, 2))

        best = None
        expected_rows = 2 ** len(species_cols)

        all_pair_combos = itertools.product(*(value_pairs_by_col[col] for col in species_cols))
        for pair_selection in all_pair_combos:
            selected = {col: pair for col, pair in zip(species_cols, pair_selection)}

            mask = np.ones(len(df), dtype=bool)
            for col, pair in selected.items():
                mask &= df[col].isin(pair).to_numpy()

            subset = df.loc[mask].copy()

            if len(subset) != expected_rows:
                continue
            if not subset["Completed"].all():
                continue

            total_steps_sum = float(subset["Total Steps"].sum())
            accepted_steps_sum = float(subset["Accepted Steps"].sum())
            tie_break_tfinal = float(subset["t_final (s)"].sum())

            # Objective: low total/accepted steps, high t_final.
            score = (total_steps_sum, accepted_steps_sum, -tie_break_tfinal)

            if best is None or score < best["score"]:
                best = {
                    "selected": selected,
                    "subset": subset.sort_values(["Total Steps", "Accepted Steps", "t_final (s)"], ascending=[True, True, False]).reset_index(drop=True),
                    "score": score,
                    "tfinal_sum": tie_break_tfinal,
                }

        return best

    best_two = pick_best_two_concentrations_per_species(df_res, species_cols)

    print("\nBest 2-concentration sweep per species (all pairwise combinations solve):")
    if best_two is None:
        print("No feasible 2x2x... subset found where every combination completed under the current caps.")
    else:
        for col in species_cols:
            pair = best_two["selected"][col]
            print(f"{col}: {pair[0]} and {pair[1]}")

        print(f"Total-steps objective (sum over all combinations): {best_two['score'][0]:.0f}")
        print(f"Accepted-steps objective (sum over all combinations): {best_two['score'][1]:.0f}")
        print(f"t_final objective (sum over all combinations, higher is better): {best_two['tfinal_sum']:.1f} s")
        print("Rows in chosen subset (sorted by low steps, high t_final):")
        display(best_two["subset"])

else:
    print("No caps set: unconstrained sweep finished.")
    display(df_res.head(8))